In [282]:
# SPDX-License-Identifier: MIT
# Copyright (c) 2025 Hammerheads Engineers sp. z o.o.
# Author: Aleksander Stanik
import sys
import os
import time
import yaml
import json
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

current_dir = os.getcwd()
repo_root = os.path.abspath(os.path.join(current_dir, '..'))

if repo_root not in sys.path:
    sys.path.append(repo_root)


import spx_python
spx_python.set_global_transparent(False)
# Initialize HTTP-based SPX client wrapper pointing to local SPX server
product_key = os.environ['SPX_PRODUCT_KEY']
wrapper = spx_python.init(address='http://localhost:8000',
                                product_key=product_key)

In [283]:
# Create a new model for the PT100 sensor
pt_100_yaml = '''
attributes:
  temperature: 0.0
import:
    /app/extensions/py_temp_sensor.py:
        class: PyTempSensor
        init:
            kwargs:
                start: 25.0
                drift: 0.0
        attributes:
            temperature: { property: temperature }
        methods: 
            run: tick
'''

# Parse YAML and build the model
data = yaml.safe_load(pt_100_yaml)
wrapper["models"]["pt_100_py"] = data
wrapper["instances"]["test_pt_100_py"] = "pt_100_py"

instance = wrapper["instances"]["test_pt_100_py"]
instance["polling"].disable()
instance.prepare()

print("Available models:", wrapper["models"].keys())
print("Available instances:", wrapper["instances"].keys())

Available models: ['pt_100_py']
Available instances: ['test_pt_100_py']


In [414]:
import plotly.graph_objects as go

# --- simulation & sampling ---
STEPS = 1500        # number of ticks
DT = 0.1           # optional: time step used for the X axis (seconds)
times, temps = [], []
temp_attr = instance["attributes"]["temperature"]
temp_attr.internal_value = 50.0  # initial temperature

for step in range(STEPS):
    instance.run()                 # advance the model by one tick
    times.append(step * DT)        # time axis (seconds)
    temps.append(temp_attr.internal_value)  # read current temperature
    # time.sleep(DT)               # uncomment for real delays

# --- plot with Plotly ---
fig = go.Figure()
fig.add_trace(go.Scatter(x=times, y=temps, mode="lines", name="Temperature"))
fig.update_layout(
    title="Temperature over Time",
    xaxis_title="Time [s]",
    yaxis_title="Temperature",
    template="plotly_white",
)
fig.show()